In [ ]:
# Install libraries
#import tabula
import os
import pandas as pd
import numpy as np
import re
import glob
from datetime import datetime
import sys
import csv
import openpyxl
import matplotlib.pyplot as plt


# Set Working Directory
os.chdir(r'C:/Users/luke.gordon/OneDrive - Resource Systems Group, Inc\Sharepoint - Model Development/CCRPC (VT) Model Update/Data/Household-Travel-Data/')




## Function Set

In [55]:
# Transpose and clean table function (combines fields with common column names)
def transpose_tbl_combinefields(in_tbl):
    out_tbl = in_tbl.set_index("Label (Grouping)").T   # Transposes the table and sets the index to the "Label (Grouping)" column
    out_tbl.columns = out_tbl.columns.str.strip().str.replace(r'\s+', ' ', regex=True) # Strips out extra spaces in the column names
    out_tbl = out_tbl.replace(',', '', regex=True)
    out_tbl = out_tbl.apply(pd.to_numeric, errors="coerce")    # Convert values from strings to numbers
    out_tbl = out_tbl.T.groupby(level=0).sum().T  # Groups columns with the same name and sums them together
    return out_tbl

def transpose_tbl(in_tbl):
    out_tbl = in_tbl.set_index("Label (Grouping)").T   # Transposes the table and sets the index to the "Label (Grouping)" column
    out_tbl.columns = out_tbl.columns.str.strip().str.replace(r'\s+', ' ', regex=True) # Strips out extra spaces in the column names
    out_tbl = out_tbl.replace(',', '', regex=True)
    out_tbl = out_tbl.apply(pd.to_numeric, errors="coerce")    # Convert values from strings to numbers
    return out_tbl





## Processing functions

In [ ]:
def total_hhs_processing(in_tbl,out_tbl):


    # Import and transpose HH size to get total households by tract
    hhs_df = in_tbl
    df_transposed = transpose_tbl(hhs_df)
    df_transposed = df_transposed.rename(columns={'Total:': 'Total_HHs'}).reset_index()

    # Change the "Label (Grouping)" field to "Blockgroup" field
    df_transposed = df_transposed.rename(columns={'index': 'Blockgroup'})

    # Remove unnecessary columns and keep only the "Blockgroup" and "Total_HHs" columns
    df_transposed = df_transposed[['Blockgroup', 'Total_HHs']]


    df_transposed.to_csv(out_tbl, index=False)

In [ ]:

def hh_size_processing(in_tbl,out_tbl,avg_over_7_per_hh):

    # Import HH Size table
    df = in_tbl


    # Transpose and clean table
    df_transposed = pd.DataFrame()
    df_transposed = transpose_tbl_size(df).reset_index()

    # Create fields by number of people in the household
    df_transposed['1_Per_HHs']  = df_transposed['1-person household']
    df_transposed['2_Per_HHs']  = df_transposed['2-person household']
    df_transposed['3_Per_HHs']  = df_transposed['3-person household']
    df_transposed['4_Per_HHs']  = df_transposed['4-person household']
    df_transposed['5_Per_HHs']  = df_transposed['5-person household']
    df_transposed['6_Per_HHs']  = df_transposed['6-person household']
    df_transposed['7+_Per_HHs'] = df_transposed['7-or-more person household']
    df_transposed['Total_HHs']  = (df_transposed['1-person household'] + df_transposed['2-person household'] + df_transposed['3-person household'] 
                                     +  df_transposed['4-person household'] + df_transposed['5-person household'] + df_transposed['6-person household'] 
                                     +  df_transposed['7-or-more person household'])


    # Drop all fields except for the new fields and the "Label (Grouping)" field
    df_transposed = df_transposed[['index', '1_Per_HHs', '2_Per_HHs', '3_Per_HHs', '4_Per_HHs', '5_Per_HHs', '6_Per_HHs', '7+_Per_HHs', 'Total_HHs']]
    # Change the "Label (Grouping)" field to "Blockgroup" field
    df_transposed = df_transposed.rename(columns={'index': 'Blockgroup'})


    # Calculate average household size by blockgroup
    avg_7_per_hh = avg_over_7_per_hh  # Average household size for households with 7 or more people
    hh_count_cols = ['1_Per_HHs', '2_Per_HHs', '3_Per_HHs', '4_Per_HHs','5_Per_HHs', '6_Per_HHs', '7+_Per_HHs', 'Total_HHs']

    df_transposed[hh_count_cols] = df_transposed[hh_count_cols].apply(
        lambda col: pd.to_numeric(
            col.astype(str).str.replace(',', '', regex=False),
            errors='coerce'
        )
    )

    df_transposed['Avg_HH_Size'] = (
        df_transposed['1_Per_HHs'] * 1
        + df_transposed['2_Per_HHs'] * 2
        + df_transposed['3_Per_HHs'] * 3
        + df_transposed['4_Per_HHs'] * 4
        + df_transposed['5_Per_HHs'] * 5
        + df_transposed['6_Per_HHs'] * 6
        + df_transposed['7+_Per_HHs'] * avg_7_per_hh
    ) / df_transposed['Total_HHs'].replace(0, np.nan)

    # Drop NaN rows
    df_transposed = df_transposed.dropna(subset=['Avg_HH_Size'])

    # Calculate 4+ person households by blockgroup
    df_transposed['4+_Per_HHs'] = df_transposed['4_Per_HHs'] + df_transposed['5_Per_HHs'] + df_transposed['6_Per_HHs'] + df_transposed['7+_Per_HHs']

    # Calculate distribution of household sizes by blockgroup
    df_transposed['1_Per_HHs_Distribution'] = df_transposed['1_Per_HHs'] / df_transposed['Total_HHs']
    df_transposed['2_Per_HHs_Distribution'] = df_transposed['2_Per_HHs'] / df_transposed['Total_HHs']
    df_transposed['3_Per_HHs_Distribution'] = df_transposed['3_Per_HHs'] / df_transposed['Total_HHs']
    df_transposed['4+_Per_HHs_Distribution'] = df_transposed['4+_Per_HHs'] / df_transposed['Total_HHs']

    # Drop unnecessary columns
    df_transposed = df_transposed[['Blockgroup', 'Total_HHs', '1_Per_HHs_Distribution', '2_Per_HHs_Distribution', '3_Per_HHs_Distribution', '4+_Per_HHs_Distribution']]


    df_transposed.to_csv(out_tbl, index=False)








In [ ]:

def hh_inc_processing(in_tbl,out_tbl,break_1,break_2,break_3,county_mean_inc):

    # Import HH Size table
    df = in_tbl


    # Transpose and clean table
    df_transposed = pd.DataFrame()
    df_transposed = transpose_tbl(df).reset_index()


    # Define county mean income
    county_mean_income = county_mean_inc

    # Define break points for income groups
    inc_group1 = break_1
    inc_group2 = break_2
    inc_group3 = break_3

    # Map original ACS income columns to their income bounds
    income_bins = {
        'Less than $10,000': (0, 10000),
        '$10,000 to $14,999': (10000, 15000),
        '$15,000 to $19,999': (15000, 20000),
        '$20,000 to $24,999': (20000, 25000),
        '$25,000 to $29,999': (25000, 30000),
        '$30,000 to $34,999': (30000, 35000),
        '$35,000 to $39,999': (35000, 40000),
        '$40,000 to $44,999': (40000, 45000),
        '$45,000 to $49,999': (45000, 50000),
        '$50,000 to $59,999': (50000, 60000),
        '$60,000 to $74,999': (60000, 75000),
        '$75,000 to $99,999': (75000, 100000),
        '$100,000 to $124,999': (100000, 125000),
        '$125,000 to $149,999': (125000, 150000),
        '$150,000 to $199,999': (150000, 200000),
        '$200,000 or more': (200000, float('inf'))
    }


    # Calculate income index for each blockgroup
    income_values = {
        col: (low + high) / 2 if high != float('inf') else 250000
        for col, (low, high) in income_bins.items()
    }
    # Calculate average household income for each row
    income_cols = list(income_bins.keys())
    df_transposed['Avg_HH_Income'] = (
        df_transposed[income_cols]
        .mul(pd.Series(income_values), axis=1)
        .sum(axis=1)
        / df_transposed[income_cols].sum(axis=1)
    )
    # Income index: county average household income = 1.0
    df_transposed['INC_INDEX'] = (
        df_transposed['Avg_HH_Income'] / county_mean_income
    )



    # Combine incomes into 4 groups based on the defined break points
    def combine_income_bins(df, lower, upper):
        cols = [
            col for col, (bin_low, bin_high) in income_bins.items()
            if bin_low >= lower and bin_high <= upper
        ]
        return df[cols].sum(axis=1)
    # Create income groups
    breaks = [0, inc_group1, inc_group2, inc_group3, float('inf')]
    for lower, upper in zip(breaks[:-1], breaks[1:]):

        if lower == 0:
            name = f'<${upper/1000:g}k'
        elif upper == float('inf'):
            name = f'${lower/1000:g}k+'
        else:
            name = f'${lower/1000:g}k-${upper/1000:g}k'

        df_transposed[name] = combine_income_bins(
            df_transposed, lower, upper
        )

    df_transposed['Total_HHs']  = (df_transposed['Less than $10,000'] + df_transposed['$10,000 to $14,999'] + df_transposed['$15,000 to $19,999']
                                     +  df_transposed['$20,000 to $24,999'] + df_transposed['$25,000 to $29,999'] + df_transposed['$30,000 to $34,999']
                                     +  df_transposed['$35,000 to $39,999'] + df_transposed['$40,000 to $44,999'] + df_transposed['$45,000 to $49,999']
                                     +  df_transposed['$50,000 to $59,999'] + df_transposed['$60,000 to $74,999'] + df_transposed['$75,000 to $99,999']
                                     +  df_transposed['$100,000 to $124,999'] + df_transposed['$125,000 to $149,999'] + df_transposed['$150,000 to $199,999'] 
                                     + df_transposed['$200,000 or more'])
    

    # Drop all fields except for the new fields and the "Label (Grouping)" field
    df_transposed = df_transposed[['index', f'<${inc_group1/1000:g}k', f'${inc_group1/1000:g}k-${inc_group2/1000:g}k', 
                                            f'${inc_group2/1000:g}k-${inc_group3/1000:g}k', f'${inc_group3/1000:g}k+','Total_HHs','Avg_HH_Income','INC_INDEX']]    
    # Change the "Label (Grouping)" field to "Blockgroup" field
    df_transposed = df_transposed.rename(columns={'index': 'Blockgroup'})
    
    
    # Calculate distribution of household incomes by blockgroup
    df_transposed['Inc1_Distribution'] = df_transposed[f'<${inc_group1/1000:g}k'] / df_transposed['Total_HHs']
    df_transposed['Inc2_Distribution'] = df_transposed[f'${inc_group1/1000:g}k-${inc_group2/1000:g}k'] / df_transposed['Total_HHs']
    df_transposed['Inc3_Distribution'] = df_transposed[f'${inc_group2/1000:g}k-${inc_group3/1000:g}k'] / df_transposed['Total_HHs']
    df_transposed['Inc4_Distribution'] = df_transposed[f'${inc_group3/1000:g}k+'] / df_transposed['Total_HHs']


    # Drop unnecessary columns
    df_transposed = df_transposed[['Blockgroup', 'Total_HHs', 'Inc1_Distribution', 'Inc2_Distribution', 'Inc3_Distribution', 'Inc4_Distribution']]


    df_transposed.to_csv(out_tbl, index=False)




In [80]:

def hh_worker_processing(in_tbl,out_tbl):

    # Import HH Workers table
    df = in_tbl

    # Transpose and clean worker table
    df_transposed = pd.DataFrame()
    df_transposed = transpose_tbl(df).reset_index()

    # Drop extra (duplicate) worker value fields
    df_transposed = df_transposed.iloc[:,0:6]

    # Change the "Label (Grouping)" field to "Blockgroup" field
    df_transposed = df_transposed.rename(columns={'index':'Tract','Total:': 'Total_HHs'})

    # Calculate distribution of household workers by tract
    df_transposed['0_Wrker_HHs_Distribution']  = df_transposed['No workers'] / df_transposed['Total_HHs']
    df_transposed['1_Wrker_HHs_Distribution']  = df_transposed['1 worker'] / df_transposed['Total_HHs']
    df_transposed['2_Wrker_HHs_Distribution']  = df_transposed['2 workers'] / df_transposed['Total_HHs']
    df_transposed['3+_Wrker_HHs_Distribution'] = df_transposed['3 or more workers'] / df_transposed['Total_HHs']

    # Drop unnecessary columns
    df_transposed = df_transposed[['Tract', 'Total_HHs', '0_Wrker_HHs_Distribution', '1_Wrker_HHs_Distribution', '2_Wrker_HHs_Distribution', '3+_Wrker_HHs_Distribution']]


    df_transposed.to_csv(out_tbl, index=False)

In [ ]:
def hh_children_processing(in_tbl,out_tbl):

    # Import HH Workers table
    df = in_tbl

    # Transpose and clean worker table
    df_transposed = pd.DataFrame()
    df_transposed = transpose_tbl(df).reset_index()

    # Change the "Label (Grouping)" field to "Blockgroup" field
    df_transposed = df_transposed.rename(columns={'index':'Blockgroup','Total:': 'Total_HHs'})

    # Calculate distribution of household workers by tract
    df_transposed['NoChildren_Proportion']  = df_transposed['Households with no people under 18 years:'] / df_transposed['Total_HHs']
    df_transposed['1+_Children_Proportion'] = df_transposed['Households with one or more people under 18 years:'] / df_transposed['Total_HHs']

    # Drop unnecessary columns
    df_transposed = df_transposed[['Blockgroup', 'Total_HHs', 'NoChildren_Proportion', '1+_Children_Proportion']]


    df_transposed.to_csv(out_tbl, index=False)

In [84]:
# Run functions

# Total HHs (Tract)
hhs_in  = pd.read_csv(os.getcwd() + '/ACS/ACS_2024_B11016_HHSize_Tracts.csv')
hhs_out = os.getcwd() + '/Outputs/TotalHHs_Tracts.csv'
total_hhs_processing(hhs_in, hhs_out)

# HH Size (BlockGroup)
hhsize_in  = pd.read_csv(os.getcwd() + '/ACS/ACS_2024_B11016_HHSize_BGs.csv')
hhsize_out = os.getcwd() + '/Outputs/HHSize_BG.csv'
avg_7_per_hh = 7.2  # Average household size for households with 7 or more people
hh_size_processing(hhsize_in, hhsize_out, avg_7_per_hh)

# HH Income (BlockGroup)
hhinc_in  = pd.read_csv(os.getcwd() + '/ACS/ACS_2024_B19001_HHInc.csv')
hhinc_out = os.getcwd() + '/Outputs/HHIncome_BG.csv'
break_1 = 25000
break_2 = 50000
break_3 = 100000
county_mean_inc = 120511
hh_inc_processing(hhinc_in, hhinc_out, break_1, break_2, break_3, county_mean_inc)

# HH Workers (Tract)
hhworker_in  = pd.read_csv(os.getcwd() + '/ACS/ACS_2024_B08202_HHWorkers.csv')
hhworker_out = os.getcwd() + '/Outputs/HHWorkers_Tracts.csv'
hh_worker_processing(hhworker_in, hhworker_out)

# HH Children (BlockGroup)
hhchildren_in  = pd.read_csv(os.getcwd() + '/ACS/ACS_2024_B11005_HHChildren.csv')
hhchildren_out = os.getcwd() + '/Outputs/HHChildren_BG.csv'
hh_children_processing(hhchildren_in, hhchildren_out)



Label (Grouping)                                         Blockgroup  \
0                 Block Group 1; Census Tract 1; Chittenden Coun...   
1                 Block Group 2; Census Tract 1; Chittenden Coun...   
2                 Block Group 3; Census Tract 1; Chittenden Coun...   
3                 Block Group 1; Census Tract 2; Chittenden Coun...   
4                 Block Group 2; Census Tract 2; Chittenden Coun...   

Label (Grouping)  Total_HHs  \
0                       871   
1                       524   
2                       501   
3                       489   
4                       817   

Label (Grouping)  Households with one or more people under 18 years:  \
0                                                               207    
1                                                               132    
2                                                               104    
3                                                                73    
4                         